# RAG: Baseline (Bi-Encoder) vs. Re-Ranking (Cross-Encoder)

**Document:** IFAB *Laws of the Game 2026/27* (single-page edition, 260 pages, effective 1 July 2026).

**Pipelines (same chunked corpus for both):**
1. **Baseline** – bi-encoder (`all-MiniLM-L6-v2`) + FAISS → Top 10 → Top 3 straight to the LLM → *Answer 1*
2. **Re-rank** – same Top 10 → cross-encoder (`ms-marco-MiniLM-L-6-v2`) → re-sorted Top 3 → same LLM prompt → *Answer 2*

The LLM (Gemini) is instructed to answer **only** from the retrieved chunks, so any difference between Answer 1 and Answer 2 comes from retrieval, not from the model's own football knowledge.

## 1. Setup

In [ ]:
# Run once if packages are missing (CPU-only torch is enough for these small models):
# !pip install torch --index-url https://download.pytorch.org/whl/cpu
# !pip install pymupdf sentence-transformers faiss-cpu google-genai python-dotenv pandas numpy

In [ ]:
import os, re, time, textwrap, urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import pymupdf                      # PDF parsing (PyMuPDF)
import faiss                        # vector store
from sentence_transformers import SentenceTransformer, CrossEncoder
from google import genai
from google.genai import types
from dotenv import load_dotenv

pd.set_option("display.max_colwidth", 120)

# ---------------------------------------------------------------------------
# API KEY: put your Gemini key in a file called `.env` next to this notebook:
#     GEMINI_API_KEY=your-key-here
# (.env is git-ignored so the key never ends up on GitHub.)
# Alternatively: os.environ["GEMINI_API_KEY"] = "your-key-here"
# ---------------------------------------------------------------------------
load_dotenv()

PDF_URL  = "https://www.theifab.com/downloads/laws-of-the-game-202627-single-pages?l=en"
PDF_PATH = Path("data/LOTG_2026-27_single.pdf")

BI_ENCODER    = "sentence-transformers/all-MiniLM-L6-v2"
CROSS_ENCODER = "cross-encoder/ms-marco-MiniLM-L-6-v2"
LLM_MODEL     = "gemini-3.5-flash"

TOP_K_RETRIEVE = 10   # candidates from the bi-encoder
TOP_K_CONTEXT  = 3    # chunks passed to the LLM

## 2. Load the PDF
The booklet may not be redistributed, so it is downloaded from the IFAB site instead of being committed to the repo.

In [ ]:
if not PDF_PATH.exists():
    PDF_PATH.parent.mkdir(exist_ok=True)
    req = urllib.request.Request(PDF_URL, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req) as r, open(PDF_PATH, "wb") as f:
        f.write(r.read())

doc = pymupdf.open(PDF_PATH)
print(f"{PDF_PATH.name}: {doc.page_count} pages")

## 3. Parsing
- Page numbers in the booklet match PDF page numbers, so each page is tagged with its section using the table of contents (pp. 6–7).
- Cover, contents and divider/photo pages are skipped (< 15 words).
- The running footer and page number at the bottom of every page are dropped.
- PyMuPDF leaves some layout characters (`\x07`, en-spaces) which are cleaned out.

In [ ]:
# (start page, section name) taken from the booklet's contents page
SECTIONS = [
    (9,   "About the Laws"),
    (16,  "Notes on the Laws of the Game"),
    (20,  "General modifications"),
    (22,  "Time-limited substitution protocol"),
    (24,  "Off-field treatment and assessment protocol"),
    (26,  "Throw-in and goal-kick countdown protocol"),
    (28,  "'Only the captain' guidelines"),
    (32,  "Guidelines for temporary dismissals (sin bins)"),
    (36,  "Guidelines for return substitutes"),
    (38,  "Additional permanent concussion substitutions protocol"),
    (45,  "Law 1 - The Field of Play"),
    (55,  "Law 2 - The Ball"),
    (59,  "Law 3 - The Players"),
    (67,  "Law 4 - The Players' Equipment"),
    (73,  "Law 5 - The Referee"),
    (85,  "Law 6 - The Other Match Officials"),
    (93,  "Law 7 - The Duration of the Match"),
    (97,  "Law 8 - The Start and Restart of Play"),
    (101, "Law 9 - The Ball in and out of Play"),
    (103, "Law 10 - Determining the Outcome of a Match"),
    (109, "Law 11 - Offside"),
    (115, "Law 12 - Fouls and Misconduct"),
    (131, "Law 13 - Free Kicks"),
    (135, "Law 14 - The Penalty Kick"),
    (141, "Law 15 - The Throw-in"),
    (145, "Law 16 - The Goal Kick"),
    (149, "Law 17 - The Corner Kick"),
    (152, "Video assistant referee (VAR) protocol"),
    (162, "FIFA Quality Programme"),
    (166, "Law changes 2026/27 - Outline summary"),
    (170, "Law changes 2026/27 - Details of all Law changes"),
    (188, "Law changes 2026/27 - Editorial changes"),
    (194, "Glossary"),
    (208, "Practical guidelines for match officials"),
]
FIRST_PAGE, LAST_PAGE = 9, 257   # everything before p.9 is cover/contents, after p.257 is blank


def section_for(page_no):
    name = None
    for start, sec in SECTIONS:
        if page_no >= start:
            name = sec
    return name


def clean(text):
    text = text.replace("\x07", "").replace("\u2002", " ").replace("\t", " ").replace("\u00ad", "")
    text = re.sub(r"-\n(?=[a-z])", "", text)   # re-join words hyphenated across lines
    text = re.sub(r"\s*\n\s*", " ", text)      # a PDF block = one paragraph
    return re.sub(r"\s{2,}", " ", text).strip()


def split_long(text, limit=150):
    """Split an unusually long paragraph into groups of sentences of <= limit words."""
    if len(text.split()) <= limit:
        return [text]
    out, cur = [], ""
    for sent in re.split(r"(?<=[.;])\s+", text):
        if cur and len((cur + " " + sent).split()) > limit:
            out.append(cur)
            cur = sent
        else:
            cur = (cur + " " + sent).strip()
    return out + [cur] if cur else out


def extract_paragraphs(doc):
    paras = []
    for page_no in range(FIRST_PAGE, LAST_PAGE + 1):
        page = doc[page_no - 1]
        if len(page.get_text().split()) < 15:          # divider / photo pages
            continue
        height = page.rect.height
        for x0, y0, x1, y1, text, *_ in page.get_text("blocks", sort=True):
            if y0 > height - 40:                       # footer + page number
                continue
            text = clean(text)
            if not text or text.isdigit():
                continue
            for piece in split_long(text):
                paras.append({"page": page_no, "section": section_for(page_no), "text": piece})
    return paras


paragraphs = extract_paragraphs(doc)
print(f"{len(paragraphs)} paragraphs extracted")

## 4. Chunking
Paragraphs are packed into chunks of up to ~160 words, never crossing a section boundary, with ~40 words of overlap
(the last paragraph(s) of a chunk are repeated at the start of the next one).
160 words keeps most chunks inside MiniLM's 256-token input limit.

The section name is prepended to the text that gets embedded (e.g. `Law 11 - Offside: ...`) so a chunk still makes sense out of context.

In [ ]:
MAX_WORDS, OVERLAP_WORDS = 160, 40


def is_heading(text):
    return len(text.split()) <= 8 and not text.endswith((".", ":", ";"))


def build_chunks(paras, max_words=MAX_WORDS, overlap_words=OVERLAP_WORDS):
    by_section = {}
    for p in paras:
        by_section.setdefault(p["section"], []).append(p)

    chunks = []
    for section, ps in by_section.items():
        cur, cur_words = [], 0

        def flush():
            if cur and not all(is_heading(c["text"]) for c in cur):
                chunks.append({"section": section,
                               "pages": sorted({c["page"] for c in cur}),
                               "text": " ".join(c["text"] for c in cur)})

        for p in ps:
            n = len(p["text"].split())
            # close the chunk when full (but never right after a heading)
            if cur and cur_words + n > max_words and not is_heading(cur[-1]["text"]):
                flush()
                carry, w = [], 0
                for c in reversed(cur):                # overlap
                    cw = len(c["text"].split())
                    if w + cw > overlap_words:
                        break
                    carry.insert(0, c)
                    w += cw
                cur, cur_words = carry, w
            cur.append(p)
            cur_words += n
        flush()

    for i, c in enumerate(chunks):
        c["id"] = i
        c["embed_text"] = f"{c['section']}: {c['text']}"
    return chunks


chunks = build_chunks(paragraphs)
chunk_df = pd.DataFrame(chunks)
chunk_df["words"] = chunk_df["text"].str.split().str.len()
print(f"{len(chunks)} chunks | words per chunk: min {chunk_df.words.min()}, "
      f"mean {chunk_df.words.mean():.0f}, max {chunk_df.words.max()}")
chunk_df[["id", "section", "pages", "words", "text"]].sample(5, random_state=0)

## 5. Pipeline 1 – Bi-Encoder index (FAISS)

In [ ]:
bi_encoder = SentenceTransformer(BI_ENCODER, device="cpu")

t0 = time.perf_counter()
embeddings = bi_encoder.encode([c["embed_text"] for c in chunks], batch_size=32,
                               normalize_embeddings=True, show_progress_bar=True)
print(f"Embedded {len(chunks)} chunks in {time.perf_counter() - t0:.1f}s, dim = {embeddings.shape[1]}")

# Vectors are L2-normalised, so inner product == cosine similarity
index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings.astype("float32"))


def retrieve(query, k=TOP_K_RETRIEVE):
    """Baseline: top-k chunks by cosine similarity. Returns list of (chunk_id, score)."""
    q = bi_encoder.encode([query], normalize_embeddings=True).astype("float32")
    scores, ids = index.search(q, k)
    return list(zip(ids[0].tolist(), scores[0].tolist()))

## 6. Pipeline 2 – Cross-Encoder re-ranker

In [ ]:
cross_encoder = CrossEncoder(CROSS_ENCODER, device="cpu")


def rerank(query, candidates):
    """Score every (query, chunk) pair jointly and re-sort.
    Returns list of (chunk_id, ce_score, original_rank)."""
    pairs = [(query, chunks[cid]["embed_text"]) for cid, _ in candidates]
    scores = cross_encoder.predict(pairs)
    order = np.argsort(-scores)
    return [(candidates[j][0], float(scores[j]), j + 1) for j in order]

## 7. LLM + grounded prompt
The same prompt template is used for both pipelines; only the three context chunks differ.

In [ ]:
client = genai.Client(api_key=os.environ.get("GEMINI_API_KEY"))

SYSTEM_PROMPT = """You are answering questions about the IFAB Laws of the Game 2026/27.
Rules:
- Use ONLY the context passages provided. Do NOT use any prior knowledge of football rules,
  even if you believe the context is incomplete or the rules used to be different.
- If the context does not contain the answer, reply exactly: "Not found in the provided context."
- Cite the passage numbers you used, e.g. [C2].
- Be concise (max 4 sentences)."""


def build_prompt(query, chunk_ids):
    context = "\n\n".join(
        f"[C{n}] ({chunks[cid]['section']}, p. {chunks[cid]['pages'][0]})\n{chunks[cid]['text']}"
        for n, cid in enumerate(chunk_ids, 1))
    return f"Context:\n{context}\n\nQuestion: {query}\nAnswer:"


def generate(query, chunk_ids):
    resp = client.models.generate_content(
        model=LLM_MODEL,
        contents=build_prompt(query, chunk_ids),
        config=types.GenerateContentConfig(system_instruction=SYSTEM_PROMPT, temperature=0),
    )
    return resp.text.strip()

## 8. Running both pipelines

In [ ]:
def run_pipelines(query, with_llm=True):
    """Runs baseline and re-rank pipelines for one query, timing each stage."""
    t0 = time.perf_counter()
    base = retrieve(query)                                   # bi-encoder top 10
    t_retrieve = time.perf_counter() - t0

    t0 = time.perf_counter()
    reranked = rerank(query, base)                           # cross-encoder over the same 10
    t_rerank = time.perf_counter() - t0

    base_top3 = [cid for cid, _ in base[:TOP_K_CONTEXT]]
    rr_top3   = [cid for cid, _, _ in reranked[:TOP_K_CONTEXT]]

    out = {"query": query, "base": base, "reranked": reranked,
           "base_top3": base_top3, "rr_top3": rr_top3,
           "t_retrieve": t_retrieve, "t_rerank": t_rerank}

    if with_llm:
        t0 = time.perf_counter(); out["answer1"] = generate(query, base_top3); out["t_llm1"] = time.perf_counter() - t0
        t0 = time.perf_counter(); out["answer2"] = generate(query, rr_top3);   out["t_llm2"] = time.perf_counter() - t0
    return out


def rank_table(res):
    """Side-by-side view of the 10 candidates: baseline rank vs cross-encoder rank."""
    ce_rank = {cid: r for r, (cid, _, _) in enumerate(res["reranked"], 1)}
    ce_score = {cid: s for cid, s, _ in res["reranked"]}
    rows = [{"chunk": cid, "section": chunks[cid]["section"], "page": chunks[cid]["pages"][0],
             "bi_rank": r, "bi_score": round(s, 3),
             "ce_rank": ce_rank[cid], "ce_score": round(ce_score[cid], 2),
             "shift": r - ce_rank[cid], "preview": chunks[cid]["text"][:90]}
            for r, (cid, s) in enumerate(res["base"], 1)]
    return pd.DataFrame(rows)


def ask(query, with_llm=True):
    res = run_pipelines(query, with_llm)
    print("Q:", query)
    display(rank_table(res))
    if with_llm:
        print("\nAnswer 1 (baseline top 3: %s):" % res["base_top3"])
        print(textwrap.fill(res["answer1"], 110))
        print("\nAnswer 2 (re-ranked top 3: %s):" % res["rr_top3"])
        print(textwrap.fill(res["answer2"], 110))
    return res


_ = ask("Can a goal be scored directly from a kick-off?", with_llm=False)

## 9. Failure-mode queries
Each query has a `gold` regex that identifies the chunk containing the decisive rule, so we can track where it ranks
before and after re-ranking. (Draft list, to be refined.)

In [ ]:
FAILURE_QUERIES = [
    {"type": "Keyword trap",
     "query": "Can a goal be scored directly from a kick-off?",
     "gold": r"goal may be scored directly against the opponents from the kick-off"},
    {"type": "Exception / corner case",
     "query": "What happens if the ball goes straight into the kicker's own goal from the kick-off?",
     "gold": r"directly enters the kicker.s goal, a corner kick is awarded"},
    {"type": "Negation / role limit",
     "query": "Can a replaced goalkeeper take a kick in the penalty shoot-out?",
     "gold": r"replaced goalkeeper takes no further part and may not take a kick"},
    {"type": "Buried chunk",
     "query": "When does the referee's authority to show cards begin and end?",
     "gold": r"power to show yellow or red cards"},
    {"type": "Negation",
     "query": "When is a player in an offside position not penalised for being offside?",
     "gold": r"It is not an offence to be in an offside position"},
    {"type": "Exception",
     "query": "What happens if a team has fewer than seven players during penalties?",
     "gold": r"must not abandon the match if a team is reduced to fewer than seven"},
    {"type": "New 2026/27 rule (grounding check)",
     "query": "What happens if the goalkeeper holds the ball in their hands for too long?",
     "gold": r"more than eight seconds before releasing"},
    {"type": "Document-specific (deleted text in 'Law changes')",
     "query": "Is the kicker cautioned if both the penalty taker and the goalkeeper commit an offence at the same time?",
     "gold": r"kicker is cautioned"},
]


def gold_ids(pattern):
    return {c["id"] for c in chunks if re.search(pattern, c["text"], re.I | re.S)}


def best_rank(ranked_ids, gold):
    ranks = [r for r, cid in enumerate(ranked_ids, 1) if cid in gold]
    return min(ranks) if ranks else None    # None = not in the top 10


def rank_shift_summary(with_llm=False):
    rows, results = [], []
    for fq in FAILURE_QUERIES:
        res = run_pipelines(fq["query"], with_llm)
        gold = gold_ids(fq["gold"])
        b = best_rank([cid for cid, _ in res["base"]], gold)
        c = best_rank([cid for cid, _, _ in res["reranked"]], gold)
        rows.append({"type": fq["type"], "query": fq["query"], "gold_chunks": sorted(gold),
                     "bi_rank": b, "ce_rank": c,
                     "in_top3_baseline": b is not None and b <= TOP_K_CONTEXT,
                     "in_top3_rerank": c is not None and c <= TOP_K_CONTEXT})
        results.append(res)
    return pd.DataFrame(rows), results


summary_df, _ = rank_shift_summary(with_llm=False)
summary_df

### 9.1 Answer 1 vs Answer 2 for each failure query

In [ ]:
failure_results = []
for fq in FAILURE_QUERIES:
    print("=" * 110)
    print(f"[{fq['type']}]")
    failure_results.append(ask(fq["query"]))

## 10. Latency profiling
Each stage is timed over several runs after a warm-up. LLM time is reported separately because it dominates and is
identical in structure for both pipelines.

In [ ]:
def profile_latency(queries, runs=5):
    for q in queries[:2]:                       # warm-up
        rerank(q, retrieve(q))
    rows = []
    for q in queries:
        for _ in range(runs):
            t0 = time.perf_counter(); cands = retrieve(q); t1 = time.perf_counter()
            rerank(q, cands);                            t2 = time.perf_counter()
            rows.append({"query": q, "bi_encoder_ms": (t1 - t0) * 1000, "cross_encoder_ms": (t2 - t1) * 1000})
    df = pd.DataFrame(rows)
    df["pipeline2_total_ms"] = df.bi_encoder_ms + df.cross_encoder_ms
    return df


lat = profile_latency([fq["query"] for fq in FAILURE_QUERIES])
latency_table = pd.DataFrame({
    "Pipeline 1 (bi-encoder retrieval)":       lat.bi_encoder_ms.describe()[["mean", "50%", "max"]],
    "Re-rank step only (cross-encoder, 10 pairs)": lat.cross_encoder_ms.describe()[["mean", "50%", "max"]],
    "Pipeline 2 (retrieval + re-rank)":         lat.pipeline2_total_ms.describe()[["mean", "50%", "max"]],
}).T.round(1).rename(columns={"50%": "median"})
latency_table.columns = [f"{c} (ms)" for c in latency_table.columns]
latency_table

In [ ]:
# End-to-end including the LLM call (from the failure-query runs above)
pd.DataFrame([{
    "query": r["query"][:60],
    "retrieval_ms": r["t_retrieve"] * 1000,
    "rerank_ms": r["t_rerank"] * 1000,
    "llm_answer1_s": r["t_llm1"],
    "llm_answer2_s": r["t_llm2"],
} for r in failure_results]).round(2)

In [ ]:
# Why not cross-encode everything? Cost of scoring the whole corpus for one query:
q = FAILURE_QUERIES[0]["query"]
t0 = time.perf_counter()
cross_encoder.predict([(q, c["embed_text"]) for c in chunks], batch_size=32)
full_ce = time.perf_counter() - t0
print(f"Cross-encoder over all {len(chunks)} chunks: {full_ce * 1000:.0f} ms "
      f"(vs {lat.cross_encoder_ms.mean():.0f} ms for 10 candidates)")

## 11. Try your own question

In [ ]:
_ = ask("Can a player be offside if they receive the ball directly from a throw-in?")